# 🥇 Gold — Daily revenue

**Purpose.** Build a daily-grain aggregate of the Silver yellow-taxi data — one row per calendar day, with revenue, trip counts, average fare, and average distance.

**Source:** `workspace.silver.yellow_taxi`
**Output:** `workspace.gold.daily_revenue`

**Why this aggregate exists.** Dashboards and BI tools should never scan 5.4M raw rows to draw a daily revenue line. The Gold layer pre-aggregates the answer so dashboard queries are fast and cheap. One row per day × two months ≈ ~60 rows.

In [0]:
# Configuration -----------------------------------------------------------------
SOURCE_TABLE = "workspace.silver.yellow_taxi"
TARGET_CATALOG = "workspace"
TARGET_SCHEMA = "gold"
TARGET_TABLE = "daily_revenue"
FULL_TABLE_NAME = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}"

# Ensure the Gold schema exists.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")
spark.sql(f"USE {TARGET_CATALOG}.{TARGET_SCHEMA}")

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {FULL_TABLE_NAME}")

In [0]:
from pyspark.sql.functions import col, to_date, count, sum as spark_sum, avg, round as spark_round

silver_df = spark.table(SOURCE_TABLE)

daily_df = (
    silver_df
    .withColumn("trip_date", to_date(col("tpep_pickup_datetime")))
    .groupBy("trip_date")
    .agg(
        count("*").alias("trip_count"),
        spark_round(spark_sum("fare_amount"), 2).alias("total_fare"),
        spark_round(spark_sum("tip_amount"), 2).alias("total_tips"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance_miles"),
        spark_round(avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
    )
    .orderBy("trip_date")
)

print(f"Daily aggregate: {daily_df.count()} rows")
display(daily_df.limit(10))

In [0]:
(
    daily_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

print(f"✓ Wrote {daily_df.count()} rows to {FULL_TABLE_NAME}")

# Quick verification — read back and show a summary.
result = spark.table(FULL_TABLE_NAME)
print(f"\nDate range: {result.agg({'trip_date': 'min'}).first()[0]} → {result.agg({'trip_date': 'max'}).first()[0]}")
print(f"Total revenue across all days: ${result.agg({'total_revenue': 'sum'}).first()[0]:,.2f}")
print(f"Busiest day:")
display(result.orderBy(col("trip_count").desc()).limit(1))

In [0]:
from pyspark.sql.functions import col, to_date, year, month

silver = spark.table("workspace.silver.yellow_taxi")

print("Rows per year-month in Silver (should all be 2024-01 or 2024-02):")
display(
    silver
    .withColumn("y", year("tpep_pickup_datetime"))
    .withColumn("m", month("tpep_pickup_datetime"))
    .groupBy("y", "m")
    .count()
    .orderBy("y", "m")
)